In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_recall_curve, classification_report, confusion_matrix

#Load Data

train_df = pd.read_csv("dataset-train-vf.csv")
test_df  = pd.read_csv("dataset-test-vf.csv")

y = (train_df["y"] == "circle").astype(int)   # convert to 0/1
X = train_df.drop(columns=["y", "ID"])
test_ids = test_df["ID"]
X_test = test_df.drop(columns=["ID"])

numeric_features = ["f1","f2","f3","f4","f5","f6","f7","f8","f9","f10"]
categorical_features = ["f11"]

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

#XGBoost Model 
xgb_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("clf", xgb_model)
])

param_grid = {
    "clf__n_estimators": [300, 500, 800],
    "clf__max_depth": [4, 6, 8],
    "clf__learning_rate": [0.01, 0.05, 0.1],
    "clf__subsample": [0.8, 1.0],
    "clf__colsample_bytree": [0.8, 1.0],
    "clf__scale_pos_weight": [5, 10, 15, 20]   # imbalance handling
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid = GridSearchCV(
    pipe,
    param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

grid.fit(X, y)

print("\nBest params:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

best_model = grid.best_estimator_

#validation + Threshold Optimization
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, stratify=y, test_size=0.2, random_state=42
)

best_model.fit(X_tr, y_tr)

val_proba = best_model.predict_proba(X_val)[:,1]
prec, rec, th = precision_recall_curve(y_val, val_proba)
f1_scores = 2 * (prec * rec) / (prec + rec + 1e-9)

best_th = th[f1_scores.argmax()]

print("\nOptimized Threshold:", best_th)
print("Optimized val F1:", f1_scores.max())

val_pred_opt = (val_proba >= best_th).astype(int)

print("\nClassification Report:\n", classification_report(y_val, val_pred_opt))
print("Confusion Matrix:\n", confusion_matrix(y_val, val_pred_opt))

#test Set Prediction
test_proba = best_model.predict_proba(X_test)[:,1]
test_pred = (test_proba >= best_th).astype(int)

submission = pd.DataFrame({
    "ID": test_ids.astype(int),
    "y": test_pred.astype(int)
})

submission.to_csv("xgb_nosmote_submission2.csv", index=False)
print("\nSaved xgb_nosmote_submission2.csv")
print(submission.head())

#try differnt thresholds
thresholds = [0.420]#best one i found for kaggle
print("Will generate submissions for thresholds:\n", thresholds)

for t in thresholds:
    preds = (test_proba >= t).astype(int)

    submission = pd.DataFrame({
        "ID": test_ids,
        "y": preds
    })

    file_name = f"xgb_thr_{t:.3f}.csv"
    submission.to_csv(file_name, index=False)
    print("Saved:", file_name)

print("\n==============================")
print("All threshold submissions generated.")

Fitting 3 folds for each of 432 candidates, totalling 1296 fits

Best params: {'clf__colsample_bytree': 0.8, 'clf__learning_rate': 0.1, 'clf__max_depth': 8, 'clf__n_estimators': 800, 'clf__scale_pos_weight': 5, 'clf__subsample': 1.0}
Best CV F1: 0.6783766119794553

Optimized Threshold: 0.21927653
Optimized val F1: 0.6406249995019531

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.97      0.97       836
           1       0.60      0.68      0.64        60

    accuracy                           0.95       896
   macro avg       0.79      0.83      0.81       896
weighted avg       0.95      0.95      0.95       896

Confusion Matrix:
 [[809  27]
 [ 19  41]]

Saved xgb_nosmote_submission2.csv
     ID  y
0  4481  0
1  4482  0
2  4483  0
3  4484  0
4  4485  0
Will generate submissions for thresholds:
 [0.42]
Saved: xgb_thr_0.420.csv

All threshold submissions generated.
